In [2]:
## put in python script for running 

import argparse
import scanpy as sc
import fastools as ft
import numpy as np
import rapids_singlecell as rsc
import numpy as np
import jax
import torch
import jax.numpy as jnp 

import cupy as cp
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from scipy import sparse
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

def filter_method(adata = None, method = ["max_counts", "min_cells"], min_cells = 3):
    if method == "max_counts":
        return filter_method_max_counts(adata)
    elif method == "min_cells":
        return filter_method_min_cells(adata, min_cells = min_cells)
    else:
        raise ValueError(f"Unknown method={method!r}.")
    
    
def filter_method_max_counts(adata):
    # filter genes less than 3 counts
    max_counts_per_gene = np.max(adata.X, axis=0)
    
    max_counts_per_gene = max_counts_per_gene.toarray().flatten()
    
    gene_mask = max_counts_per_gene >= 3
    
    print(f'Number of genes to keep = {gene_mask.sum()}')
    
    # subset AnnData object
    return adata[:, gene_mask].copy()

def filter_method_min_cells(adata, min_cells = 3):
    rsc.get.anndata_to_GPU(adata)
    rsc.pp.filter_genes(adata, min_cells = min_cells)
    rsc.get.anndata_to_CPU(adata)
    return adata


parser = argparse.ArgumentParser()
parser.add_argument(
    "--method",
    type=str,
    choices=["max_counts", "min_cells"],
    default="min_cells",
    help="Filtering method to use: 'max_counts' or 'min_cells' (default: min_cells)"
    )
parser.add_argument(
    "--cell_type",
    type=str,
    nargs="+",
    required=True,
    help="Celltypes to calculate, space-separated (e.g., HSC MPP)"
    )
parser.add_argument(
    "--path_dir",
    type=str,
    help="Folder to store results (e.g., ../results/normal_HSC/pseudobulk/)"
    )

# to run in cluster
# Parse arguments
# args = parser.parse_args()

# to run in jupyter

from argparse import Namespace

args = Namespace(
        # method = "min_cells",
        method = "max_counts", 
        # cell_type = ['HSC', 'MPP'],
        cell_type = ["B"],
        # path_dir = "/home/lel2/luan/projects/cell_tissue_phenotype/running_slurm/test/",
        path_dir = "/home/lel2/luan/projects/cell_tissue_phenotype/results/normal_HSC/pseudobulk/max_counts_2/"
        )
    
# use harmony-corrected, 5000 top variable genes 
# our_cdata = sc.read("/data1/soldatr/luan/projects/cell_tissue_phenotype/results/our_cdata_5000.h5ad")

# use top 5000 variable genes, raw counts with the paper annotation
# our_cdata = sc.read("/data1/soldatr/luan/projects/cell_tissue_phenotype/results/our_cdata_raw_counts_5000.h5ad")

# use all cells with raw counts
our_cdata = sc.read("/data1/soldatr/luan/projects/cell_tissue_phenotype/results/our_cdata_raw_celltypes.h5ad")

# filter genes or cells 
our_cdata = filter_method(adata = our_cdata, method = args.method)

# remove some of the cells with abnormal read counts
gene_counts_per_cell = our_cdata.X.sum(axis = 1).A1
q_low, q_high = np.percentile(gene_counts_per_cell, [5, 95])
print(f"5 percentile is {q_low}, 95 percentile is {q_high}")
our_cdata = our_cdata[(gene_counts_per_cell > q_low) & (gene_counts_per_cell < q_high)]

df_covariate = pd.read_csv("/data1/soldatr/luan/projects/cell_tissue_phenotype/results/normal_covariates.csv")

our_cdata.obs = (
    our_cdata.obs.reset_index() # index (cell barcodes) becomes a column called 'index'
    .merge(df_covariate, on = "indiv_id", how = "left") #merge
    .set_index("index") # restore original row names as index
    )
our_cdata.obs.index.name = None
our_cdata.obs.head()

#combined sexes and src into one column
our_cdata.obs['sex_src'] = our_cdata.obs['sexes'].astype(str) + '_' + our_cdata.obs['src'].astype(str)
our_cdata.obs['sex_src'] = our_cdata.obs['sex_src'].astype("category")

# saving counts data
our_cdata.layers['counts'] = our_cdata.X.copy()
our_cdata.layers['counts'].data

our_cdata.obs['sample_id'] = our_cdata.obs['exp_name'].astype('string').str.cat(our_cdata.obs['indiv_id'].astype('string'), sep = '_')
our_cdata.obs['sample_id'] = our_cdata.obs['sample_id'].astype('category')

# add batch keys
our_cdata.obs['batch_key'] = our_cdata.obs['sample_id'].astype('string').str.cat(our_cdata.obs['sex_src'].astype('string'), sep = '_')
our_cdata.obs['batch_key'] = our_cdata.obs['batch_key'].astype('category')

# # note that there are NA cell_type, but ignore here
# # 1️⃣ Check if any column has missing values
# our_cdata.obs.isna().sum()

# # 2️⃣ Show only columns that contain at least one NA
# print(our_cdata.obs.columns[our_cdata.obs.isna().any()].tolist())

# # 3️⃣ Show rows that contain any NA
# rows_with_na = our_cdata.obs[our_cdata.obs.isna().any(axis=1)]


# # Step 1: Add "Unknown" as a new category
# our_cdata_org.obs['cell_type'] = our_cdata_org.obs['cell_type'].cat.add_categories("Unknown")
# # replace NA with unknown
# our_cdata_org.obs.loc[our_cdata_org.obs.isna().any(axis=1), "cell_type"] = "Unknown"



Number of genes to keep = 17573
5 percentile is 751.0, 95 percentile is 8448.0


exp_name                     0
barcode                      0
num_umis_orig                0
num_umis_fil                 0
is_doublet                   0
mito_frac                    0
is_ultima                    0
indiv_id                     0
src                          0
metacell                     0
dissolved                    0
metacell_level               0
cells_rare_gene_module       0
rare_cell                    0
metacell_name                0
most_similar                 0
date                         0
cell_type                 5599
ages                         0
sexes                        0
sex_src                      0
sample_id                    0
batch_key                    0
dtype: int64

In [8]:
import pandas as pd
import os 

org_dir = "/home/lel2/luan/projects/cell_tissue_phenotype/results/normal_HSC/pseudobulk/refined/"
method = "max_counts"

cell_type_proportions_df = pd.read_csv(
        # "/data1/soldatr/luan/projects/cell_tissue_phenotype/results/normal_HSC/cell_type_proportions_harmony_sample_id.csv",
        # "/data1/soldatr/luan/projects/cell_tissue_phenotype/results/normal_HSC/cell_type_proportions_harmony_sample_id_avg_pseudobulk.csv",
        # "/data1/soldatr/luan/projects/cell_tissue_phenotype/results/normal_HSC/pseudobulk/max_counts/parallel/cell_type_proportions_pseudobulk_max_counts.csv",
        # "/home/lel2/luan/projects/cell_tissue_phenotype/results/normal_HSC/pseudobulk/weighted_ncells/cell_type_proportions_pseudobulk_max_counts.csv",
        os.path.join(org_dir, f"{method}/mean_gene_expression/cell_type_proportions_pseudobulk_{method}.csv"),
        index_col=0
    )

assert set(our_cdata.obs['sample_id'].cat.categories.tolist()) == set(cell_type_proportions_df.index.tolist())


In [11]:
from joblib import Parallel, delayed

def extract(sample_id):
    subset = our_cdata[
        (our_cdata.obs['sample_id'] == sample_id) &
        (our_cdata.obs['cell_type'].isin(args.cell_type))
    ].copy()
    return subset.X

Xs_raw = Parallel(n_jobs=5)(
    delayed(extract)(sid)
    for sid in our_cdata.obs['sample_id'].cat.categories.tolist()
)


In [22]:
import pickle
import os


with open(os.path.join(os.getcwd(), f"data/{'_'.join(args.cell_type)}_Xs_samples_raw.pkl"), 'wb') as f:
    pickle.dump(Xs_raw, f)

# Load later with:
# with open('Xs_samples.pkl', 'rb') as f:
#     Xs_loaded = pickle.load(f)


In [23]:
with open(os.path.join(os.getcwd(), f"data/{'_'.join(args.cell_type)}_Xs_samples_raw.pkl"), 'rb') as f:
    Xs_loaded = pickle.load(f)

In [11]:
from joblib import Parallel, delayed

def extract(sample_id):
    subset = our_cdata[
        (our_cdata.obs['sample_id'] == sample_id) &
        (our_cdata.obs['cell_type'].isin(['HSC', 'MPP']))
    ].copy()
    return subset.X

Xs_raw = Parallel(n_jobs=5)(
    delayed(extract)(sid)
    for sid in our_cdata.obs['sample_id'].cat.categories.tolist()
)


In [19]:
import pickle
import os


with open(os.path.join(os.getcwd(), "deep_learning/Xs_samples_raw.pkl"), 'wb') as f:
    pickle.dump(Xs_raw, f)

# Load later with:
# with open('Xs_samples.pkl', 'rb') as f:
#     Xs_loaded = pickle.load(f)


In [ ]:
def log_normalize(our_cdata_org, target_sum = 1e4):
    # rsc.get.anndata_to_GPU(our_cdata_org)
    # rsc.pp.normalize_total(our_cdata_org, target_sum = target_sum)
    # rsc.pp.log1p(our_cdata_org)
    # rsc.get.anndata_to_CPU(our_cdata_org)

    sc.pp.normalize_total(our_cdata_org, target_sum = target_sum)
    sc.pp.log1p(our_cdata_org)

    return our_cdata_org
    

In [ ]:
from joblib import Parallel, delayed

def extract_and_normalize(sample_id):
    subset = our_cdata[
        (our_cdata.obs['sample_id'] == sample_id) &
        (our_cdata.obs['cell_type'].isin(['HSC', 'MPP']))
    ].copy()
    return log_normalize(subset).X

Xs = Parallel(n_jobs=5)(
    delayed(extract_and_normalize)(sid)
    for sid in our_cdata.obs['sample_id'].cat.categories.tolist()
)


In [ ]:
# Xs = []
# for sample_id in our_cdata.obs['sample_id'].cat.categories.tolist():
#     # sample_id = our_cdata.obs['sample_id'].cat.categories.tolist()[10]
    
#     # our_cdata_obs_subset = our_cdata.obs.loc[(our_cdata.obs['sample_id'] == sample_id) & (our_cdata.obs['cell_type'].isin(['HSC', 'MPP']))]
#     # our_cdata_subset = our_cdata[our_cdata_obs_subset.index.tolist(), :].copy()
#     # a better way
#     # our_cdata_subset = our_cdata[(our_cdata.obs['sample_id'] == sample_id)].copy()
#     our_cdata_subset_HSC_MPP = our_cdata[(our_cdata.obs['sample_id'] == sample_id) & (our_cdata.obs['cell_type'].isin(['HSC', 'MPP']))].copy()
#     # our_cdata_subset_HSC_MPP = log_normalize(our_cdata_subset_HSC_MPP)
#     Xs.append(our_cdata_subset_HSC_MPP.X)


In [ ]:
#the order in Xs is similar to the one in cell type proportions

import pandas as pd
import os 

org_dir = "/home/lel2/luan/projects/cell_tissue_phenotype/results/normal_HSC/pseudobulk/refined/"
method = "max_counts"

cell_type_proportions_df = pd.read_csv(
        # "/data1/soldatr/luan/projects/cell_tissue_phenotype/results/normal_HSC/cell_type_proportions_harmony_sample_id.csv",
        # "/data1/soldatr/luan/projects/cell_tissue_phenotype/results/normal_HSC/cell_type_proportions_harmony_sample_id_avg_pseudobulk.csv",
        # "/data1/soldatr/luan/projects/cell_tissue_phenotype/results/normal_HSC/pseudobulk/max_counts/parallel/cell_type_proportions_pseudobulk_max_counts.csv",
        # "/home/lel2/luan/projects/cell_tissue_phenotype/results/normal_HSC/pseudobulk/weighted_ncells/cell_type_proportions_pseudobulk_max_counts.csv",
        os.path.join(org_dir, f"{method}/mean_gene_expression/cell_type_proportions_pseudobulk_{method}.csv"),
        index_col=0
    )

assert(cell_type_proportions_df.index.tolist() == our_cdata.obs['sample_id'].cat.categories.tolist())


In [ ]:
import pickle

with open('Xs_samples.pkl', 'wb') as f:
    pickle.dump(Xs, f)

# Load later with:
# with open('Xs_samples.pkl', 'rb') as f:
#     Xs_loaded = pickle.load(f)
